# Entraînement du Temporal Fusion Transformer

Ce notebook détaille toutes les étapes pour entraîner le modèle Temporal Fusion Transformer sur les jeux de données pickle présents dans le dossier `datasets/`.


## Pré-requis

- Installez les dépendances du projet (via `poetry install` ou `pip install -r requirements.txt`) avant d'exécuter ce notebook.
- Placez-vous à la racine du dépôt ou définissez la variable d'environnement `TRADER_AUTO_ROOT` pointant vers cette racine.
- Vérifiez que les datasets d'entraînement et de validation se trouvent dans le dossier `datasets/` et suivent le préfixe attendu (`full_dataset_focus_train_` / `full_dataset_focus_val_`).
- Pour l'exécution sur Google Colab, uploadez le dépôt, montez Google Drive puis adaptez la cellule de configuration du chemin de projet si nécessaire.
- Laissez la cellule _Synchronisation GitHub_ récupérer automatiquement la branche souhaitée.
- Configurez `TRADER_AUTO_PAT` / `GITHUB_TOKEN` (et éventuellement `TRADER_AUTO_USER`) si vous devez cloner un dépôt privé.


## Synchronisation GitHub

Cette section garantit que le notebook pointe toujours vers la dernière version du code publiée sur GitHub.

- `TRADER_AUTO_REPO` (facultatif) permet de changer l'URL du dépôt.
- `TRADER_AUTO_BRANCH` (facultatif) permet de choisir la branche à suivre.
- `TRADER_AUTO_CLONE_DIR` (facultatif) impose un chemin de clonage spécifique s'il est différent de la racine courante.
- `TRADER_AUTO_PAT` ou `GITHUB_TOKEN` (facultatif) peut contenir un jeton d'accès personnel pour cloner un dépôt privé.
- `TRADER_AUTO_USER` (facultatif) précise le nom d'utilisateur Git associé au jeton si nécessaire.

Si un dépôt est déjà présent localement, la cellule suivante effectue un `git fetch` suivi d'un `git pull --ff-only` sur la branche choisie.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from urllib.parse import urlparse, urlunparse

REPO_URL = os.environ.get(
    "TRADER_AUTO_REPO", "https://github.com/clementremillieux/trader.git"
)
REPO_BRANCH = os.environ.get("TRADER_AUTO_BRANCH", "crypto_V3")
TOKEN = os.environ.get("TRADER_AUTO_PAT") or os.environ.get("GITHUB_TOKEN")
TOKEN_USER = os.environ.get("TRADER_AUTO_USER", "oauth2")

if "COLAB_GPU" in os.environ:
    default_clone_dir = Path("/content/trader_auto")
else:
    default_clone_dir = Path(os.environ.get("TRADER_AUTO_CLONE_DIR", Path.cwd()))

# Détection d'un dépôt existant dans l'arborescence courante
existing_repo_root = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / ".git").exists():
        existing_repo_root = candidate
        break

clone_dir = (
    (existing_repo_root or Path(os.environ.get("TRADER_AUTO_ROOT", default_clone_dir)))
    .expanduser()
    .resolve()
)
clone_dir.parent.mkdir(parents=True, exist_ok=True)

parsed_url = urlparse(REPO_URL)
authed_url = REPO_URL
display_url = REPO_URL
masked_token = "***"

if TOKEN and parsed_url.scheme.startswith("http"):
    userinfo = f"{TOKEN_USER}:{TOKEN}" if TOKEN_USER else TOKEN
    authed_netloc = "@".join(filter(None, [userinfo, parsed_url.netloc]))
    authed_url = urlunparse(parsed_url._replace(netloc=authed_netloc))
    masked_userinfo = f"{TOKEN_USER}:{masked_token}" if TOKEN_USER else masked_token
    masked_netloc = "@".join(filter(None, [masked_userinfo, parsed_url.netloc]))
    display_url = urlunparse(parsed_url._replace(netloc=masked_netloc))

# Si le dossier existe déjà mais n'est pas un dépôt git, on le supprime pour repartir proprement
if clone_dir.exists() and not (clone_dir / ".git").exists():
    entries = list(clone_dir.glob("*"))
    if entries:
        print(
            f"Le dossier {clone_dir} existe sans dépôt Git. Suppression du contenu pour relancer le clone."
        )
    shutil.rmtree(clone_dir)

print(f"Synchronisation du dépôt {display_url} sur la branche {REPO_BRANCH}")
print(f"Répertoire cible: {clone_dir}")


def sanitize_cmd(cmd: list[str]) -> str:
    cmd_display = " ".join(cmd)
    if TOKEN:
        cmd_display = cmd_display.replace(TOKEN, masked_token)
    return cmd_display


def run_git(cmd, cwd=None, env=None):
    env = env or os.environ.copy()
    cmd_display = sanitize_cmd([str(part) for part in cmd])
    print(f"$ {cmd_display}")
    completed = subprocess.run(
        cmd,
        check=True,
        cwd=cwd,
        text=True,
        capture_output=True,
        env=env,
    )
    stdout = completed.stdout.strip()
    stderr = completed.stderr.strip()
    if stdout:
        print(stdout)
    if stderr:
        print(stderr)
    return completed


try:
    if (clone_dir / ".git").exists():
        if TOKEN:
            run_git(["git", "remote", "set-url", "origin", authed_url], cwd=clone_dir)
        else:
            run_git(["git", "remote", "set-url", "origin", REPO_URL], cwd=clone_dir)
        run_git(["git", "fetch", "origin", REPO_BRANCH], cwd=clone_dir)
        run_git(["git", "checkout", REPO_BRANCH], cwd=clone_dir)
        run_git(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=clone_dir)
        if TOKEN:
            run_git(["git", "remote", "set-url", "origin", REPO_URL], cwd=clone_dir)
    else:
        run_git(
            ["git", "clone", "--branch", REPO_BRANCH, authed_url, str(clone_dir)],
            cwd=clone_dir.parent,
        )
        if TOKEN:
            run_git(["git", "remote", "set-url", "origin", REPO_URL], cwd=clone_dir)
except subprocess.CalledProcessError as exc:
    stderr = exc.stderr.strip() if exc.stderr else ""
    stdout = exc.stdout.strip() if exc.stdout else ""
    if TOKEN:
        stderr = stderr.replace(TOKEN, masked_token)
        stdout = stdout.replace(TOKEN, masked_token)
    diagnostic = "\n".join(
        part
        for part in [
            "Commande échouée: " + sanitize_cmd(exc.cmd),
            f"Code retour: {exc.returncode}",
            f"stdout:\n{stdout}" if stdout else "",
            f"stderr:\n{stderr}" if stderr else "",
        ]
        if part
    )
    raise RuntimeError(
        "La synchronisation GitHub a échoué. Vérifiez l'URL, la branche, les droits réseau ou les conflits locaux.\n"
        + diagnostic
    ) from exc

commit = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=clone_dir)
    .decode()
    .strip()
)
os.environ["TRADER_AUTO_ROOT"] = str(clone_dir)
os.chdir(clone_dir)

print(f"Synchronisation terminée. Commit actif: {commit}")


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path(os.environ.get("TRADER_AUTO_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "scripts").exists():
    raise RuntimeError(
        f"Impossible de localiser le dossier scripts depuis {PROJECT_ROOT}. \\n
        "Définissez la variable d'environnement TRADER_AUTO_ROOT avant d'exécuter ce notebook."
    )
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Racine du projet: {PROJECT_ROOT}")
print(f"Dossier datasets: {PROJECT_ROOT / 'datasets'}")


In [ ]:
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from scripts.train_tft import (
    TFTConfig,
    TemporalFusionTransformer,
    compute_statistics,
    estimate_num_samples,
    list_dataset_files,
    prepare_dataloader,
    setup_logging,
    train,
)


In [ ]:
training_params = {
    "train_prefix": "full_dataset_focus_train_",
    "val_prefix": "full_dataset_focus_val_",
    "max_train_files": None,  # Ajustez pour un entrainement rapide (ex: 5)
    "max_val_files": None,
    "limit_samples_per_file": None,  # Limiter le nb de séquences par fichier si besoin
    "stat_sample_fraction": 0.2,
    "batch_size": 64,
    "epochs": 5,
    "learning_rate": 3e-4,
    "grad_clip": 1.0,
    "lambda_reg": 0.3,
    "lambda_vol": 0.05,
    "seed": 42,
    "num_workers": 2,
    "mixed_precision": torch.cuda.is_available(),
    "save_every": 0,
}
output_dir = PROJECT_ROOT / "models" / "tft_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(training_params, indent=2, ensure_ascii=False))
print(f"Les checkpoints seront sauvegardés dans: {output_dir}")


In [ ]:
train_files = list_dataset_files(
    training_params["train_prefix"], training_params["max_train_files"]
)
val_files = list_dataset_files(
    training_params["val_prefix"], training_params["max_val_files"]
)
print(f"Fichiers train: {len(train_files)}")
print(f"Fichiers val: {len(val_files)}")
train_files[:3], val_files[:3]


In [ ]:
stats = compute_statistics(
    file_paths=train_files,
    sample_fraction=training_params["stat_sample_fraction"],
    seed=training_params["seed"],
    limit_samples_per_file=training_params["limit_samples_per_file"],
)
print(f"Dimension des features: {stats.feature_dim}")
print(f"Longueur de séquence: {stats.seq_len}")
print(f"Vocabulaire tau: {stats.tau_vocab_size}")
print("Poids de classes:", stats.class_weights)
print("Reg mean/std:", stats.reg_mean, stats.reg_std)
print("Vol mean/std:", stats.vol_mean, stats.vol_std)


In [ ]:
train_loader = prepare_dataloader(
    files=train_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=True,
    shuffle_samples=True,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
)
val_loader = prepare_dataloader(
    files=val_files,
    stats=stats,
    batch_size=training_params["batch_size"],
    shuffle_files=False,
    shuffle_samples=False,
    seed=training_params["seed"],
    limit_per_file=training_params["limit_samples_per_file"],
    num_workers=training_params["num_workers"],
)

est_train = estimate_num_samples(train_files, stats)
est_val = estimate_num_samples(val_files, stats)
print(f"Séquences d'entraînement estimées: {est_train}")
print(f"Séquences de validation estimées: {est_val}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = TFTConfig(
    input_dim=stats.feature_dim,
    seq_len=stats.seq_len,
    tau_vocab_size=stats.tau_vocab_size,
    hidden_dim=512,
    num_heads=8,
    num_transformer_blocks=6,
    dropout=0.2,
    conv_kernel_sizes=(3, 5, 7),
    conv_dilations=(1, 2, 4),
    static_dim=256,
)
model = TemporalFusionTransformer(cfg).to(device)
class_weights = torch.from_numpy(stats.class_weights)
setup_logging(verbose=True)
print(model)
print(f"Device utilisé: {device}")


In [ ]:
best_metrics = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    epochs=training_params["epochs"],
    lr=training_params["learning_rate"],
    grad_clip=training_params["grad_clip"],
    lambda_reg=training_params["lambda_reg"],
    lambda_vol=training_params["lambda_vol"],
    mixed_precision=training_params["mixed_precision"],
    output_dir=output_dir,
    save_every=training_params["save_every"],
)
best_metrics


In [ ]:
artifact_path = output_dir / "final_model_notebook.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
        "normalization": {
            "mean": stats.mean.tolist(),
            "std": stats.std.tolist(),
            "class_weights": stats.class_weights.tolist(),
            "tau_vocab_size": stats.tau_vocab_size,
            "reg_mean": stats.reg_mean,
            "reg_std": stats.reg_std,
            "vol_mean": stats.vol_mean,
            "vol_std": stats.vol_std,
            "vol_log": stats.vol_log,
        },
        "best_metrics": best_metrics,
        "params": training_params,
    },
    artifact_path,
)
print(f"Modèle sauvegardé dans: {artifact_path}")


In [ ]:
if best_metrics:
    metrics_df = pd.DataFrame(best_metrics, index=[0]).T.rename(columns={0: "valeur"})
    display(metrics_df)
    if "val_confusion" in best_metrics and best_metrics["val_confusion"]:
        confusion = np.array(best_metrics["val_confusion"])
        confusion_df = pd.DataFrame(
            confusion,
            index=["réel_-1", "réel_0", "réel_1"],
            columns=["prédit_-1", "prédit_0", "prédit_1"],
        )
        display(confusion_df)
else:
    print("Aucune métrique de validation enregistrée (best_metrics est vide).")


## Prochaines étapes

- Ajustez les hyperparamètres ou les préfixes de jeu de données pour entraîner des variantes.
- Relancez l'entraînement en changeant `max_train_files` / `max_val_files` pour un smoke test rapide.
- Analysez les checkpoints générés dans `models/tft_notebook/` ou chargez-les pour de l'inférence.
